In [ ]:
import datasets
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
# import librosa
import torch

In [2]:
ds = load_dataset("facebook/voxpopuli", "fr", split="test", streaming=True)

sample = next(iter(ds))
print(sample["audio"])

In [9]:
lang="fr"
model_name = "jonatasgrosman/wav2vec2-large-xlsr-53-french"
SAMPLES=5

In [4]:
model = Wav2Vec2ForCTC.from_pretrained(model_name)
processor = Wav2Vec2Processor.from_pretrained(model_name)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [5]:
def speech_file_to_array(batch):
  speech_array = batch["audio"]["array"]
  sampling_rate = batch["audio"]["sampling_rate"]
  batch["speech"] = speech_array
  batch["sentence"] = batch["raw_text"].upper() # Use raw_text as sentence for processing
  return batch

In [6]:
test_dataset = ds.map(speech_file_to_array)

In [10]:
speech_samples = []
sentences = []

for i, sample in enumerate(test_dataset):
    if i >= SAMPLES:
        break
    speech_samples.append(sample["speech"])
    sentences.append(sample["sentence"])

inputs = processor(speech_samples, sampling_rate=16_000, return_tensors="pt", padding=True)

print("Structure of the generated inputs:")
print(inputs)
print("Number of samples processed:", len(speech_samples))
print("Example of an input_values shape:", inputs.input_values[0].shape)

Structure of the generated inputs:
{'input_values': tensor([[-0.2501, -0.0774,  0.2018,  ...,  0.0829,  0.0568,  0.0431],
        [ 0.0086,  0.0149, -0.0054,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0148, -0.0168, -0.0296,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0047,  0.0067, -0.0038,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0651,  0.0737,  0.0153,  ...,  0.0000,  0.0000,  0.0000]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)}
Number of samples processed: 5
Example of an input_values shape: torch.Size([321593])


Structure du dataset `ds` avant processing.

In [ ]:
print(next(iter(ds)))


{'audio_id': '20131024-0900-PLENARY-4-fr_20131024-10:24:36_6', 'language': 2, 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7bd53cccc4d0>, 'raw_text': 'Je souhaite juste rappeler que ces droits de plantation étaient intégrés dans la réforme de 2008, qui a été adoptée par le Conseil des ministres. Mais malgré cela et au delà de cela, dans le cadre de cette réforme 2013, nous sommes revenus sur cette question, compte tenu aussi du rapport', 'normalized_text': 'je souhaite juste rappeler que ces droits de plantation étaient intégrés dans la réforme de deux mille huit qui a été adoptée par le conseil des ministres. mais malgré cela et au delà de cela dans le cadre de cette réforme deux mille treize nous sommes revenus sur cette question compte tenu aussi du rapport', 'gender': 'male', 'speaker_id': 'None', 'is_gold_transcript': True, 'accent': 'None'}


Structure du dataset après application de `speech_file_to_array`.

In [ ]:
sample_after_processing = next(iter(test_dataset))
print(sample_after_processing)

{'audio_id': '20131024-0900-PLENARY-4-fr_20131024-10:24:36_6', 'language': 2, 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7bd53ce34bf0>, 'raw_text': 'Je souhaite juste rappeler que ces droits de plantation étaient intégrés dans la réforme de 2008, qui a été adoptée par le Conseil des ministres. Mais malgré cela et au delà de cela, dans le cadre de cette réforme 2013, nous sommes revenus sur cette question, compte tenu aussi du rapport', 'normalized_text': 'je souhaite juste rappeler que ces droits de plantation étaient intégrés dans la réforme de deux mille huit qui a été adoptée par le conseil des ministres. mais malgré cela et au delà de cela dans le cadre de cette réforme deux mille treize nous sommes revenus sur cette question compte tenu aussi du rapport', 'gender': 'male', 'speaker_id': 'None', 'is_gold_transcript': True, 'accent': 'None', 'speech': array([-0.01843262, -0.00570679,  0.01486206, ...,  0.00610352,
        0.00418091,  0.00317383], dtype=float32

In [11]:
with torch.no_grad():
  logits = model(inputs.input_values, attention_mask=inputs.attention_mask).logits

predicted_ids = torch.argmax(logits, dim=-1)
predicted_sentences = processor.batch_decode(predicted_ids)

for i, prediction in enumerate(predicted_sentences):
  print("-" * 30)
  print(f"Reference:  {sentences[i]}")
  print(f"Prediction: {prediction}")

------------------------------
Reference:  JE SOUHAITE JUSTE RAPPELER QUE CES DROITS DE PLANTATION ÉTAIENT INTÉGRÉS DANS LA RÉFORME DE 2008, QUI A ÉTÉ ADOPTÉE PAR LE CONSEIL DES MINISTRES. MAIS MALGRÉ CELA ET AU DELÀ DE CELA, DANS LE CADRE DE CETTE RÉFORME 2013, NOUS SOMMES REVENUS SUR CETTE QUESTION, COMPTE TENU AUSSI DU RAPPORT
Prediction: uête juste rappeler que ces droits dé plantation étaient intégrés dans la réforme deux mille huit qui a été votés par le conseil de ministre mais malgré cella et au-delà decella dont le cadrue de cette réforme deux mille treize sont revenus sur cette question et continu aussi de rapport
------------------------------
Reference:  LES PARLEMENTS NATIONAUX ONT ACCEPTÉ QUE LES COMPÉTENCES COMMERCIALES SOIENT EXCLUSIVEMENT EXERCÉES AU NIVEAU EUROPÉEN.
Prediction: les parlements nationaux ont accepté que les compétences commerciales soient exclusivement exercées au niveau européen
------------------------------
Reference:  C'EST CE QUE JE DEMANDE À CHACU